## 1. Setup
Import libraries and create the local output directory.


In [1]:
from pathlib import Path

import polars as pl
from pyscbwrapper import SCB

root=Path.cwd().resolve()

data_dir = root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

## 2. SCB Table Configuration
Define the table mapping and choose the target table ID.


In [2]:
TABLES = {
    "month_tab": ("en", "AM", "AM0401", "AM0401I", "NAKUSysselYrke2012M"),
}

tab_id = "month_tab"

## 3. Initialize SCB Client
Create the API client for the selected SCB table.


In [3]:
scb = SCB(*TABLES[f"{tab_id}"])

In [4]:
#scb.info()

## 4. Load Variable Metadata
Read available dimensions and extract keys/values used to build the query.


In [5]:
var_ = scb.get_variables()

In [6]:
attachment_key = next(k for k in var_ if "degree" in k.lower())
degree = var_[attachment_key][0]


dg_key = next((k for k in var_ if "degree" in k.lower()), None)

if dg_key:
    degree = var_[dg_key]

attachment_key = "".join(dg_key.split())

In [7]:
occupations_key = next(k for k in var_ if "occupation" in k.lower())
occupations = var_[occupations_key]

In [8]:
observations_key = next(k for k in var_ if "observations" in k.lower())
observations = var_[observations_key][0]

In [9]:
months_key = next(k for k in var_ if "month" in k.lower())
months = var_[months_key]

In [10]:
sex_key = next(k for k in var_ if "sex" in k.lower())
sex = var_[sex_key][:2]

## 5. Build and Run Query
Set dimension selections and fetch data from SCB.


In [11]:

scb.set_query(
    **{
        attachment_key: degree,
        occupations_key: occupations,
        months_key: months,
        observations_key: observations,
        sex_key: sex,
    },
)

In [12]:
scb_data = scb.get_data()
scb_fetch = scb_data["data"]

## 6. Create Code Mappings
Map SCB codes to readable occupation and sex labels.


In [13]:
codes = scb.get_query()["query"][1]["selection"]["values"]
occ_dict = dict(zip(codes, occupations, strict=False))

In [14]:
sex_codes = scb.get_query()["query"][4]["selection"]["values"]
sex_dict = dict(zip(sex_codes, sex, strict=False))

## 7. Transform Response to DataFrame
Normalize API payload, clean records, and cast data types.


In [15]:
df = (
    pl.DataFrame(scb_fetch)
    .with_columns([
        pl.col("key").list.get(1).alias("code_1"),
        pl.col("key").list.get(2).alias("sex"),
        pl.col("key").list.get(3).alias("month"),
        pl.col("values").list.get(0).alias("value"),
    ])
    .drop(["key", "values"])
    .with_columns([
        pl.col("code_1").replace(occ_dict).alias("occupation"),
        pl.col("sex").replace(sex_dict).alias("sex"),
    ])
    .filter(~pl.col("code_1").is_in(["0002", "0000"]))
    .with_columns([
        pl.col("code_1").cast(pl.Utf8),
        pl.col("occupation").cast(pl.Utf8),
        pl.col("sex").cast(pl.Utf8),
        pl.col("month").cast(pl.Utf8),
        pl.col("value").cast(pl.Utf8, strict=False),
    ])
)


## 8. Format Month Values
Convert month values to a readable year-month label.


In [16]:
df = df.with_columns(
    pl.col("month")
        .str.replace("M", "-")
        .str.strptime(pl.Date, "%Y-%m")
        .dt.strftime("%Y-%b")
        .alias("month"),
)


## 9. Preview Output
Check a sample of the transformed data.


In [17]:
df.head(10)

code_1,sex,month,value,occupation
str,str,str,str,str
"""1""","""men""","""2015-Jan""","""135.1""","""Managers"""
"""1""","""men""","""2015-Feb""","""125.6""","""Managers"""
"""1""","""men""","""2015-Mar""","""120.2""","""Managers"""
"""1""","""men""","""2015-Apr""","""141.6""","""Managers"""
"""1""","""men""","""2015-May""","""137.1""","""Managers"""
"""1""","""men""","""2015-Jun""","""112.8""","""Managers"""
"""1""","""men""","""2015-Jul""","""142.5""","""Managers"""
"""1""","""men""","""2015-Aug""","""137.1""","""Managers"""
"""1""","""men""","""2015-Sep""","""120.7""","""Managers"""


## 10. Save Dataset
Write the final dataset to Parquet in the `data` folder.


In [18]:
df.write_parquet(data_dir/ "scb_months.parquet")